# Audio Feature Extraction — IEMOCAP

Encoder: configurable via `MODEL_NAME` in the config cell (default: `microsoft/wavlm-large`)  
Feature: masked mean-pool over last hidden state frames → shape `(HIDDEN_SIZE,)` per utterance  
Output: `Dataset/Processed/IEMOCAP/features/audio_{MODEL_TAG}.pt`  
Format: `dict {utt_id: np.array(HIDDEN_SIZE,)}`

In [1]:
import torch
import torchaudio
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoFeatureExtractor, AutoModel
from tqdm.notebook import tqdm

In [ ]:
PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/IEMOCAP"
print(f"Data root : {DATA_ROOT}")
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 8      # keep low — variable-length audio padded per batch, 12 GB VRAM
SAMPLE_RATE = 16000  # both WavLM-Large and HuBERT-Large expect 16 kHz mono

# ── Swap model here ───────────────────────────────────────────────────────────
MODEL_NAME = "microsoft/wavlm-large"      # any HuggingFace AutoModel-compatible SSL audio encoder
# Examples:
#   "microsoft/wavlm-large"           → audio_microsoft_wavlm_large.pt       (hidden=1024)
#   "facebook/hubert-large-ll60k"     → audio_facebook_hubert_large_ll60k.pt  (hidden=1024)
#   "facebook/hubert-base-ls960"      → audio_facebook_hubert_base_ls960.pt   (hidden=768)
#   "facebook/wav2vec2-large-960h"    → audio_facebook_wav2vec2_large_960h.pt (hidden=1024)
# ─────────────────────────────────────────────────────────────────────────────
MODEL_TAG = MODEL_NAME.replace("/", "_").replace("-", "_")
print(f"Model    : {MODEL_NAME}")
print(f"Tag      : {MODEL_TAG}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [3]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Utterances : {len(labels)}")
print(f"Columns    : {labels.columns.tolist()}")
labels.head(3)

Utterances : 10039
Columns    : ['utt_id', 'session', 'dialog', 'speaker', 'emotion', 'valence', 'arousal', 'dominance', 'start_time', 'end_time', 'transcription', 'audio_path', 'video_path', 'text_path']


,utt_id,session,dialog,speaker,emotion,valence,arousal,dominance,start_time,end_time,transcription,audio_path,video_path,text_path
0,Ses01F_impro01_F000,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,6.2901,8.2357,Excuse me.,audio/Ses01F_impro01_F000.wav,video/Ses01F_impro01_F000.mp4,text/Ses01F_impro01_F000.txt
1,Ses01F_impro01_F001,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,10.0100,11.3925,Yeah.,audio/Ses01F_impro01_F001.wav,video/Ses01F_impro01_F001.mp4,text/Ses01F_impro01_F001.txt
2,Ses01F_impro01_F002,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,14.8872,18.0175,Is there a problem?,audio/Ses01F_impro01_F002.wav,video/Ses01F_impro01_F002.mp4,text/Ses01F_impro01_F002.txt


In [4]:
processor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
model = model.to(DEVICE)
HIDDEN_SIZE = model.config.hidden_size
print(f"Hidden size : {HIDDEN_SIZE}")
print(f"Params      : {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

Hidden size : 1024
Params      : 315M


In [5]:
def load_wav(path: Path) -> np.ndarray:
    """Load wav, resample to SAMPLE_RATE if needed, convert to mono numpy float32."""
    wav, sr = torchaudio.load(str(path))
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    return wav.mean(0).numpy()  # mono


def extract_batch(audio_paths: list[Path]) -> np.ndarray:
    """Returns masked mean-pooled hidden states, shape (B, HIDDEN_SIZE)."""
    waveforms = [load_wav(p) for p in audio_paths]

    inputs = processor(
        waveforms,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    hidden = out.last_hidden_state  # (B, T, H)

    # Mask-aware mean pool: exclude padding frames from the average
    if "attention_mask" in inputs:
        feat_len = model._get_feat_extract_output_lengths(
            inputs["attention_mask"].sum(-1)
        ).long()
        T = hidden.shape[1]
        frame_mask = (
            torch.arange(T, device=DEVICE).unsqueeze(0) < feat_len.unsqueeze(1)
        ).unsqueeze(-1).float()  # (B, T, 1)
        feat = (hidden * frame_mask).sum(1) / frame_mask.sum(1)  # (B, H)
    else:
        feat = hidden.mean(1)

    return feat.cpu().numpy()

In [6]:
features: dict[str, np.ndarray] = {}
skipped: list[str] = []

utt_ids     = labels["utt_id"].tolist()
audio_paths = labels["audio_path"].tolist()

batch_ids:   list[str]  = []
batch_paths: list[Path] = []

def flush_batch():
    if not batch_paths:
        return
    feats = extract_batch(batch_paths)
    for bid, feat in zip(batch_ids, feats):
        features[bid] = feat
    batch_ids.clear()
    batch_paths.clear()

for uid, apath in tqdm(zip(utt_ids, audio_paths), total=len(utt_ids), desc="Extracting"):
    p = DATA_ROOT / apath
    if not p.exists():
        skipped.append(uid)
        continue
    batch_ids.append(uid)
    batch_paths.append(p)
    if len(batch_paths) == BATCH_SIZE:
        flush_batch()

flush_batch()  # remainder

print(f"Extracted : {len(features)}")
print(f"Skipped   : {len(skipped)}  {skipped if skipped else ''}")

Extracting:   0%|          | 0/10039 [00:00<?, ?it/s]

/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Extracted : 10039
Skipped   : 0  


In [7]:
out_path = FEATURES_DIR / f"audio_{MODEL_TAG}.pt"
torch.save(features, out_path)
size_mb = out_path.stat().st_size / 1e6
print(f"Saved  : {out_path}")
print(f"Size   : {size_mb:.1f} MB")

Saved  : Dataset/Processed/IEMOCAP/features/audio_microsoft_wavlm_large.pt
Size   : 62.9 MB


In [8]:
loaded = torch.load(out_path, weights_only=False)

expected_count = len(labels) - len(skipped)
assert len(loaded) == expected_count, f"Expected {expected_count}, got {len(loaded)}"

sample_key  = next(iter(loaded))
sample_feat = loaded[sample_key]
assert sample_feat.shape == (HIDDEN_SIZE,), f"Expected ({HIDDEN_SIZE},), got {sample_feat.shape}"

all_feats = np.stack(list(loaded.values()))
has_nan   = np.isnan(all_feats).any()
has_inf   = np.isinf(all_feats).any()
assert not has_nan, "NaN detected"
assert not has_inf, "Inf detected"

l2_norm = np.linalg.norm(sample_feat)
print(f"Count   : {len(loaded)} / {len(labels)}")
print(f"Shape   : {sample_feat.shape}  ✓")
print(f"Mean    : {all_feats.mean():.4f}")
print(f"Std     : {all_feats.std():.4f}")
print(f"Has NaN : {has_nan}  ✓")
print(f"Has Inf : {has_inf}  ✓")

sample_row = labels[labels.utt_id == sample_key].iloc[0]
print(f"\nSample utt_id : {sample_key}")
print(f"Emotion       : {sample_row.emotion}")
print(f"Audio path    : {sample_row.audio_path}")
print(f"L2 norm       : {l2_norm:.2f}")

Count   : 10039 / 10039
Shape   : (1024,)  ✓
Mean    : -0.0003
Std     : 0.0822
Has NaN : False  ✓
Has Inf : False  ✓

Sample utt_id : Ses01F_impro01_F000
Emotion       : neu
Audio path    : audio/Ses01F_impro01_F000.wav
L2 norm       : 3.70
